<a href="https://colab.research.google.com/github/nicolastibata/MINE_4210_ADL_202620/blob/main/assingments/assingment_1/MINE_4210_ADL_202620_T1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![Logo ADL](https://github.com/nicolastibata/MINE_4210_ADL_202620/blob/main/docs/logo.png?raw=true)


# **Taller 1: Clasificación multiclase con un MLP**

### **Contexto y Objetivos**

La obesidad es un problema de salud pública creciente, asociado a enfermedades cardiovasculares, diabetes y otras condiciones crónicas. Identificar el nivel de riesgo de una persona a partir de sus hábitos, sin depender de mediciones clínicas directas como el peso o la estatura, puede ser útil cuando esta información no está disponible o se busca anticipar el riesgo antes de que se manifieste físicamente. En este taller deberán estimar el nivel de obesidad de una persona a partir de variables sociodemográficas y de estilo de vida, como alimentación, actividad física, consumo de alcohol y medio de transporte.

El objetivo de este Taller es construir un perceptrón multicapa (MLP) para esta tarea, siguiendo el proceso de machine learning. Se realizará una búsqueda de hiperparámetros sobre la arquitectura de la red, variando el número de capas ocultas y el número de neuronas por capa.


| Hiperparámetro | Valores para explorar |
|-|-|
| Número de capas ocultas | 1, 2 o 3. |
| Neuronas por capa oculta | 16, 32 o 64 (mismo valor en todas las capas ocultas de una misma arquitectura). |
| Callback obligatorio | Early stopping, monitoreando val_loss, con un valor de patience justificado. |


### **Entregable**

Notebook (.ipynb) con todas las celdas ejecutadas y con todas las indicaciones anteriormente estipuladas en la sección de laboratorios en **Bloque Neón**.



### **Rúbrica**

| Criterio | Peso | Qué se evalúa |
|-|-|-|
| Preprocesamiento de datos | 15 % | Codificación adecuada de las variables categóricas, diferenciando entre nominales y ordinales escalamiento de las variables numéricas; y separación correcta de los conjuntos de entrenamiento, <br> validación y prueba, evitando fuga de información. |
| Implementación del MLP | 15 % | Arquitectura correctamente construida, con función de activación de salida y función de pérdida apropiadas para clasificación multiclase; código funcional y bien organizado. |
| Búsqueda de hiperparámetros | 25 % | Las 9 configuraciones están correctamente implementadas y entrenadas, early stopping está configurado con un valor de patience justificado. Los resultados se presentan de forma <br> clara mediante una tabla o gráfico comparativo. |
| Evaluación y métricas | 15 % | Uso de métricas apropiadas para clasificación multiclase, considerando posibles diferencias en el tamaño de las clases; evaluación final realizada correctamente sobre el conjunto de test.  |
| Interpretación y justificación | 20 % | Justificación de la arquitectura final con base en los resultados y las curvas de entrenamiento y validación; identificación de señales de sobreajuste o subajuste; conclusiones sobre el efecto <br> de la profundidad y el ancho de la red. |
| Calidad y reproducibilidad del notebook | 10 % | El notebook se ejecuta de principio a fin sin errores, está organizado en un orden lógico, no contiene código innecesario e incluye explicaciones breves en Markdown que guían la lectura. |

# PREPROCESAMIENTO DE DATOS

In [22]:
import pandas as pd
import numpy as np

In [23]:
ruta = 'C:\\Users\\Diego\\Documents\\ADL\\taller-1\\talleres-deep-learning-mine-4210\\data\\obesidad.csv'
df_obesidad=pd.read_csv(ruta, sep=";")

In [24]:
df_obesidad.head()

,Genero,Edad,Antecedentes_familiares,Come_calorico,Frecuencia_verduras,Comidas_dia,Picoteo,Fuma,Agua_dia,Monitorea_calorias,Actividad_fisica,Tiempo_pantallas,Alcohol,Transporte,Nivel_obesidad
0,Femenino,21.0,Si,No,2,3,A veces,No,2,No,0,1,No,Transporte_Publico,Peso_Normal
1,Femenino,21.0,Si,No,3,3,A veces,Si,3,Si,3,0,A veces,Transporte_Publico,Peso_Normal
2,Masculino,23.0,Si,No,2,3,A veces,No,2,No,2,1,Frecuentemente,Transporte_Publico,Peso_Normal
3,Masculino,27.0,No,No,3,3,A veces,No,2,No,2,0,Frecuentemente,Caminando,Sobrepeso_Nivel_I
4,Masculino,22.0,No,No,2,1,A veces,No,2,No,0,0,A veces,Transporte_Publico,Sobrepeso_Nivel_II


In [32]:
df_obesidad.describe()

,Genero,Edad,Antecedentes_familiares,Come_calorico,Frecuencia_verduras,Comidas_dia,Picoteo,Fuma,Agua_dia,Monitorea_calorias,...,Tiempo_pantallas,Alcohol_A veces,Alcohol_Frecuentemente,Alcohol_No,Alcohol_Siempre,Transporte_Automovil,Transporte_Bicicleta,Transporte_Caminando,Transporte_Motocicleta,Transporte_Transporte_Publico
count,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,...,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000,2087.000000
mean,0.504073,24.353090,0.825108,0.883565,2.425970,2.703402,1.146143,0.021083,2.011500,0.045999,...,0.669861,0.661236,0.033541,0.304744,0.000479,0.218495,0.003354,0.026354,0.005271,0.746526
std,0.500103,6.368801,0.379966,0.320823,0.585177,0.797108,0.459494,0.143695,0.685322,0.209533,...,0.673970,0.473403,0.180088,0.460409,0.021890,0.413324,0.057831,0.160223,0.072426,0.435104
min,0.000000,14.000000,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,19.915937,1.000000,1.000000,2.000000,3.000000,1.000000,0.000000,2.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,22.847618,1.000000,1.000000,2.000000,3.000000,1.000000,0.000000,2.000000,0.000000,...,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,26.000000,1.000000,1.000000,3.000000,3.000000,1.000000,0.000000,2.000000,0.000000,...,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,61.000000,1.000000,1.000000,3.000000,4.000000,3.000000,1.000000,3.000000,1.000000,...,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [26]:
df_obesidad["Agua_dia"].value_counts(normalize=True)

Agua_dia
2    0.530426
3    0.240537
1    0.229037
Name: proportion, dtype: float64

### Codificación de variables nominales: `Alcohol` y `Transporte`

`Transporte` es una variable nominal: sus categorías (Automovil, Motocicleta, Bicicleta, Caminando, Transporte_Publico) no tienen un orden natural.

`Alcohol` (No, A veces, Frecuentemente, Siempre) sí sugiere un orden de frecuencia, pero en este taller se trata como **nominal** por el espaciado desigual e incierto entre sus categorías. Ambas se codifican con **one-hot encoding**.

In [27]:
variables_nominales = ["Alcohol", "Transporte"]

df_obesidad = pd.get_dummies(df_obesidad, columns=variables_nominales, dtype=int)

df_obesidad.head()

,Genero,Edad,Antecedentes_familiares,Come_calorico,Frecuencia_verduras,Comidas_dia,Picoteo,Fuma,Agua_dia,Monitorea_calorias,...,Nivel_obesidad,Alcohol_A veces,Alcohol_Frecuentemente,Alcohol_No,Alcohol_Siempre,Transporte_Automovil,Transporte_Bicicleta,Transporte_Caminando,Transporte_Motocicleta,Transporte_Transporte_Publico
0,Femenino,21.0,Si,No,2,3,A veces,No,2,No,...,Peso_Normal,0,0,1,0,0,0,0,0,1
1,Femenino,21.0,Si,No,3,3,A veces,Si,3,Si,...,Peso_Normal,1,0,0,0,0,0,0,0,1
2,Masculino,23.0,Si,No,2,3,A veces,No,2,No,...,Peso_Normal,0,1,0,0,0,0,0,0,1
3,Masculino,27.0,No,No,3,3,A veces,No,2,No,...,Sobrepeso_Nivel_I,0,1,0,0,0,0,1,0,0
4,Masculino,22.0,No,No,2,1,A veces,No,2,No,...,Sobrepeso_Nivel_II,1,0,0,0,0,0,0,0,1


In [28]:
df_obesidad.head()

,Genero,Edad,Antecedentes_familiares,Come_calorico,Frecuencia_verduras,Comidas_dia,Picoteo,Fuma,Agua_dia,Monitorea_calorias,...,Nivel_obesidad,Alcohol_A veces,Alcohol_Frecuentemente,Alcohol_No,Alcohol_Siempre,Transporte_Automovil,Transporte_Bicicleta,Transporte_Caminando,Transporte_Motocicleta,Transporte_Transporte_Publico
0,Femenino,21.0,Si,No,2,3,A veces,No,2,No,...,Peso_Normal,0,0,1,0,0,0,0,0,1
1,Femenino,21.0,Si,No,3,3,A veces,Si,3,Si,...,Peso_Normal,1,0,0,0,0,0,0,0,1
2,Masculino,23.0,Si,No,2,3,A veces,No,2,No,...,Peso_Normal,0,1,0,0,0,0,0,0,1
3,Masculino,27.0,No,No,3,3,A veces,No,2,No,...,Sobrepeso_Nivel_I,0,1,0,0,0,0,1,0,0
4,Masculino,22.0,No,No,2,1,A veces,No,2,No,...,Sobrepeso_Nivel_II,1,0,0,0,0,0,0,0,1


### Codificación de variables binarias

`Antecedentes_familiares`, `Come_calorico`, `Fuma`, `Monitorea_calorias` y `Genero` tienen exactamente dos categorías, así que se codifican como una sola columna 0/1 en lugar de one-hot completo (evita una columna redundante, sin introducir ninguna relación de orden porque con 2 categorías no existe ambigüedad de orden).

In [29]:
mapeo_si_no = {"No": 0, "Si": 1}
variables_binarias_si_no = ["Antecedentes_familiares", "Come_calorico", "Fuma", "Monitorea_calorias"]

for col in variables_binarias_si_no:
    df_obesidad[col] = df_obesidad[col].map(mapeo_si_no)

df_obesidad["Genero"] = df_obesidad["Genero"].map({"Femenino": 0, "Masculino": 1})

df_obesidad[variables_binarias_si_no + ["Genero"]].head()

,Antecedentes_familiares,Come_calorico,Fuma,Monitorea_calorias,Genero
0,1,0,0,0,0
1,1,0,1,1,0
2,1,0,0,0,1
3,0,0,0,0,1
4,0,0,0,0,1


### Codificación de variables ordinales: `Picoteo` y `Agua_dia`

`Picoteo` (No < A veces < Frecuentemente < Siempre) tiene un orden claro de frecuencia, así que se mapea a enteros consecutivos respetando ese orden (no se puede usar el orden alfabético porque no coincide con el orden real).

`Agua_dia` ya viene codificada como entero (1, 2, 3) representando categorías crecientes de consumo de agua, así que no requiere transformación adicional.

In [30]:
mapeo_picoteo = {"No": 0, "A veces": 1, "Frecuentemente": 2, "Siempre": 3}
df_obesidad["Picoteo"] = df_obesidad["Picoteo"].map(mapeo_picoteo)

df_obesidad[["Picoteo", "Agua_dia"]].head()

,Picoteo,Agua_dia
0,1,2
1,1,3
2,1,2
3,1,2
4,1,2


In [31]:
df_obesidad.head()

,Genero,Edad,Antecedentes_familiares,Come_calorico,Frecuencia_verduras,Comidas_dia,Picoteo,Fuma,Agua_dia,Monitorea_calorias,...,Nivel_obesidad,Alcohol_A veces,Alcohol_Frecuentemente,Alcohol_No,Alcohol_Siempre,Transporte_Automovil,Transporte_Bicicleta,Transporte_Caminando,Transporte_Motocicleta,Transporte_Transporte_Publico
0,0,21.0,1,0,2,3,1,0,2,0,...,Peso_Normal,0,0,1,0,0,0,0,0,1
1,0,21.0,1,0,3,3,1,1,3,1,...,Peso_Normal,1,0,0,0,0,0,0,0,1
2,1,23.0,1,0,2,3,1,0,2,0,...,Peso_Normal,0,1,0,0,0,0,0,0,1
3,1,27.0,0,0,3,3,1,0,2,0,...,Sobrepeso_Nivel_I,0,1,0,0,0,0,1,0,0
4,1,22.0,0,0,2,1,1,0,2,0,...,Sobrepeso_Nivel_II,1,0,0,0,0,0,0,0,1
